In [1]:
import json
m = json.load(open("model_artifacts/xgboost_model.json"))
print("model saved with xgboost:", m.get("version"))
!head -n 5 sagemaker_XGB*/inference.py

model saved with xgboost: [2, 1, 4]
%%writefile inference.py
import os
import joblib
import json
import xgboost as xgb


In [2]:
%%bash
mkdir -p cpi-deploy
cp xgboost-model.tar.gz cpi-deploy/
python -m venv sm2-env
sm2-env/bin/pip install -q "sagemaker<3"
sm2-env/bin/python -c "import sagemaker; print('SDK', sagemaker.__version__)"


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/home/sagemaker-user/sm2-env/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


SDK 2.257.7


In [3]:
%%writefile cpi-deploy/inference.py
import os, json
import joblib
import pandas as pd
import xgboost as xgb

def model_fn(model_dir):
    booster = xgb.Booster()
    booster.load_model(os.path.join(model_dir, "xgboost_model.json"))
    features = joblib.load(os.path.join(model_dir, "features.pkl"))
    return {"model": booster, "features": features}

def input_fn(request_body, request_content_type):
    if request_content_type == "application/json":
        return json.loads(request_body)
    raise ValueError(f"Unsupported content type: {request_content_type}")

def predict_fn(input_data, model_dict):
    # input: {"last_value": float, "recent_changes": [12 monthly changes, oldest first]}
    row = pd.DataFrame([input_data["recent_changes"][::-1]], columns=model_dict["features"])
    d = float(model_dict["model"].predict(xgb.DMatrix(row))[0])
    return {"predicted_change": d, "predicted_level": input_data["last_value"] + d}

def output_fn(prediction, accept):
    return json.dumps(prediction)

Writing cpi-deploy/inference.py


In [4]:
%%writefile cpi-deploy/deploy_endpoint.py
import sagemaker
from sagemaker import get_execution_role
from sagemaker.xgboost.model import XGBoostModel
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

session = sagemaker.Session()
role = get_execution_role()
s3_path = session.upload_data("xgboost-model.tar.gz",
                              bucket=session.default_bucket(),
                              key_prefix="cpi-forecaster-xgboost")

model = XGBoostModel(model_data=s3_path, role=role,
                     entry_point="inference.py",
                     framework_version="3.0-5")
predictor = model.deploy(initial_instance_count=1,
                         instance_type="ml.m5.large",
                         serializer=JSONSerializer(),
                         deserializer=JSONDeserializer())
print("Endpoint:", predictor.endpoint_name)
try:
    print(predictor.predict({"last_value": 325.0, "recent_changes": [0.3] * 12}))
finally:
    predictor.delete_endpoint()
    print("Endpoint deleted")

Writing cpi-deploy/deploy_endpoint.py


In [6]:
!cd cpi-deploy && ../sm2-env/bin/python deploy_endpoint.py

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
/home/sagemaker-user/sm2-env/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
/home/sagemaker-user/sm2-env/lib/python3.12/site-packages/sagemaker/model.py:347: SageMaker

In [7]:
   !ls -la --time-style=long-iso cpi-deploy/xgboost-model.tar.gz xgboost-model.tar.gz

-rw-r--r--. 1 sagemaker-user users 94024 2026-11-06 13:09 cpi-deploy/xgboost-model.tar.gz
-rw-r--r--. 1 sagemaker-user users 94024 2026-11-06 13:09 xgboost-model.tar.gz
